# 04 — RAG Evaluation with RAGAS

Full evaluation of the RAG pipeline using the **RAGAS** framework.

## Metrics
| Metric | What it measures |
|--------|------------------|
| **Faithfulness** | Is the answer factually grounded in the retrieved context? |
| **Answer Relevancy** | How well does the answer address the question? |
| **Context Precision** | Are the retrieved chunks actually relevant? |
| **Context Recall** | Does the retrieved context cover the ground truth? |


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from src.rag.rag_pipeline import MedicalRAGPipeline
from src.evaluation.evaluator import RAGASEvaluator
from src.utils.config import get_settings

cfg = get_settings('../configs/config.yaml')
print('Config loaded ✓')

## 1. Define Evaluation Dataset

In [ ]:
eval_dataset = [
    {
        'question': 'What does low hemoglobin indicate and what are the symptoms?',
        'ground_truth': 'Low hemoglobin indicates anemia. Symptoms include fatigue, weakness, '
                        'pale skin, shortness of breath, dizziness, and cold hands and feet.'
    },
    {
        'question': 'What is a normal HbA1c and what level indicates diabetes?',
        'ground_truth': 'Normal HbA1c is below 5.7%. Pre-diabetes is 5.7-6.4%. '
                        'Diabetes is diagnosed at HbA1c of 6.5% or higher.'
    },
    {
        'question': 'Why is high LDL cholesterol dangerous?',
        'ground_truth': 'High LDL cholesterol causes plaque buildup in arteries, increasing '
                        'risk of atherosclerosis, heart attack, and stroke.'
    },
    {
        'question': 'What causes vitamin D deficiency?',
        'ground_truth': 'Vitamin D deficiency is caused by inadequate sun exposure, '
                        'poor diet, malabsorption disorders, obesity, and darker skin.'
    },
    {
        'question': 'What does elevated serum creatinine suggest?',
        'ground_truth': 'Elevated creatinine suggests impaired kidney function and '
                        'reduced glomerular filtration rate. GFR calculation is recommended.'
    },
]
print(f'{len(eval_dataset)} evaluation samples defined')

## 2. Run RAG Pipeline on All Questions

In [ ]:
rag = MedicalRAGPipeline(cfg)

questions, answers, contexts, ground_truths = [], [], [], []
latencies = []

for sample in eval_dataset:
    rag.reset_conversation()
    resp = rag.ask(sample['question'])
    
    questions.append(sample['question'])
    answers.append(resp.answer)
    contexts.append([d['content'] for d in resp.retrieved_docs])
    ground_truths.append(sample['ground_truth'])
    latencies.append(resp.latency_ms)
    
    print(f'Q: {sample["question"][:60]}…')
    print(f'A: {resp.answer[:120]}…')
    print(f'Docs retrieved: {len(resp.retrieved_docs)} | Latency: {resp.latency_ms:.0f}ms')
    print()

print(f'Average latency: {sum(latencies)/len(latencies):.0f}ms')

## 3. Run RAGAS Evaluation

In [ ]:
evaluator = RAGASEvaluator()

try:
    metrics = evaluator.evaluate(questions, answers, contexts, ground_truths)
    print('RAGAS Metrics:')
    print(f'  Faithfulness:      {metrics.faithfulness:.4f}')
    print(f'  Answer Relevancy:  {metrics.answer_relevancy:.4f}')
    print(f'  Context Precision: {metrics.context_precision:.4f}')
    print(f'  Context Recall:    {metrics.context_recall:.4f}')
except Exception as e:
    print(f'RAGAS evaluation error: {e}')
    print('Ensure Ollama is running and the model is available.')
    metrics = None

## 4. Visualise RAGAS Metrics

In [ ]:
if metrics:
    metric_names = ['Faithfulness', 'Answer\nRelevancy', 'Context\nPrecision', 'Context\nRecall']
    scores = [
        metrics.faithfulness,
        metrics.answer_relevancy,
        metrics.context_precision,
        metrics.context_recall,
    ]
    
    # Radar chart
    fig = go.Figure(data=go.Scatterpolar(
        r=scores + [scores[0]],
        theta=['Faithfulness', 'Answer Relevancy', 'Context Precision', 'Context Recall', 'Faithfulness'],
        fill='toself',
        name='RAG Performance',
        line_color='royalblue',
    ))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title='RAGAS Evaluation Radar Chart',
        showlegend=False,
    )
    fig.write_html('../outputs/ragas_radar.html')
    fig.show()
    
    # Bar chart
    fig2, ax = plt.subplots(figsize=(8, 4))
    colors = ['green' if s >= 0.8 else 'orange' if s >= 0.6 else 'red' for s in scores]
    bars = ax.bar(metric_names, scores, color=colors, alpha=0.8)
    ax.set_ylim(0, 1.15)
    ax.axhline(y=0.8, color='green', linestyle='--', alpha=0.4, label='Good (0.8)')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.4, label='Fair (0.6)')
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{score:.3f}', ha='center', va='bottom', fontweight='bold')
    ax.set_title('RAGAS Evaluation Metrics')
    ax.set_ylabel('Score')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../outputs/ragas_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/ragas_radar.html, outputs/ragas_metrics.png')
else:
    print('No metrics to visualise.')

## 5. Latency Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(range(len(latencies)), latencies, color='steelblue', alpha=0.8)
ax.set_yticks(range(len(latencies)))
ax.set_yticklabels([f'Q{i+1}' for i in range(len(latencies))])
ax.set_xlabel('Latency (ms)')
ax.set_title(f'RAG Latency per Query (avg: {sum(latencies)/len(latencies):.0f}ms)')
ax.axvline(sum(latencies)/len(latencies), color='red', linestyle='--', label='Average')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/latency_analysis.png', dpi=150)
plt.show()